# Retail Customer Analytics: Statistical Hypothesis Testing

This notebook focuses on **Phase 11: Statistical Analysis** of the Retail Sales & Customer Intelligence project.
We will answer critical business questions using statistical hypothesis testing:
1. **Hypothesis 1**: Do high-value customers (top 50% by monetary spending) have a higher Average Order Value (AOV) compared to lower-value customers?
2. **Hypothesis 2**: Does the Average Order Value (AOV) vary significantly across different geographical regions (countries)?

We will load data directly from our **DuckDB** database, perform data validation, check statistical assumptions, run tests (T-Test and ANOVA), and interpret the business implications.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Set style for plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

### Establish Connection to DuckDB and Fetch Customer Metrics

In [ ]:
db_path = os.path.join("..", "data", "processed", "retail_analytics.db")
conn = duckdb.connect(db_path)

# Load RFM metrics alongside regional/demographic metadata
query = """
SELECT 
    customer_key,
    country,
    gender,
    marital_status,
    recency,
    frequency,
    monetary,
    -- Calculate AOV at the customer level
    monetary / frequency AS customer_aov
FROM gold.customer_rfm_base;
"""
df = conn.execute(query).fetchdf()
conn.close()

print(f"Loaded {df.shape[0]} customer records.")
df.head()

## 2. Hypothesis 1: AOV of High-Value vs. Low-Value Customers

**Business Context**: Marketing wants to know if customers who spend more in total (`monetary`) do so because they buy more expensive items per order (`higher AOV`), or simply because they shop more frequently. If AOV is significantly different, we should design different reward structures for high-value segments.

### Hypothesis Formulation
- **Null Hypothesis ($H_0$)**: The mean Average Order Value (AOV) of High-Value customers is equal to the mean AOV of Low-Value customers ($\\mu_{high} = \\mu_{low}$).
- **Alternative Hypothesis ($H_1$)**: The mean AOV of High-Value customers is different from the mean AOV of Low-Value customers ($\\mu_{high} \\neq \\mu_{low}$).

### Segment Customers
We will split customers into two groups using the median of total spending (`monetary`).

In [ ]:
median_monetary = df['monetary'].median()
df['spending_segment'] = np.where(df['monetary'] > median_monetary, 'High-Value', 'Low-Value')

# Summary statistics
summary = df.groupby('spending_segment')['customer_aov'].agg(['count', 'mean', 'median', 'std']).reset_index()
summary

### Assumption Checks
For an independent two-sample t-test, we must check:
1. **Independence**: Satisfied by the experimental/customer design.
2. **Normality**: The AOV within each group should be approximately normally distributed (or group sizes are large enough that the Central Limit Theorem holds).
3. **Homoscedasticity**: Homogeneity of variances.

In [ ]:
# Visualizing distributions
sns.displot(data=df, x='customer_aov', hue='spending_segment', kind='kde', fill=True, height=5, aspect=1.5)
plt.title("Customer AOV Distribution by Spending Segment")
plt.xlabel("AOV ($)")
plt.show()

In [ ]:
# Statistical Check for Equal Variances (Levene's Test)
high_val = df[df['spending_segment'] == 'High-Value']['customer_aov']
low_val = df[df['spending_segment'] == 'Low-Value']['customer_aov']

stat, p_val = stats.levene(high_val, low_val)
print(f"Levene's test p-value: {p_val:.4f}")
if p_val < 0.05:
    print("Warning: Variances are significantly different. We must use Welch's T-Test.")
else:
    print("Variances are equal. We can use Student's T-Test.")

### Execute T-Test
We will run Welch's t-test (`equal_var=False`) which is robust to unequal variances.

In [ ]:
t_stat, p_val = stats.ttest_ind(high_val, low_val, equal_var=False)
print(f"Welch's T-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_val:.4e}")

# Calculate Effect Size (Cohen's d)
mean_diff = high_val.mean() - low_val.mean()
pooled_std = np.sqrt((high_val.std()**2 + low_val.std()**2) / 2)
cohens_d = mean_diff / pooled_std
print(f"  Cohen's d:   {cohens_d:.4f}")

### Business Interpretation
If the p-value is $< 0.05$, we reject the null hypothesis. 
- If AOV is higher for High-Value customers: They contribute value by buying premium/bulk products.
- If AOV is similar or lower: High-Value customers are driven purely by purchasing frequency. We should focus on keeping them engaged (frequency) rather than upsells.

## 3. Hypothesis 2: AOV Differences across Regions (Countries)

**Business Context**: Supply Chain and Pricing teams want to know if customer purchase power (reflected in order sizes/AOV) varies by country. If it does, we can optimize localized pricing models and distribution hubs.

### Hypothesis Formulation
- **Null Hypothesis ($H_0$)**: The mean AOV is equal across all countries ($\\mu_1 = \\mu_2 = \\dots = \\mu_k$).
- **Alternative Hypothesis ($H_1$)**: At least one country has a mean AOV that is different from the others.

### Inspect Country Level AOV

In [ ]:
# Filter out 'Unspecified' country
df_clean = df[df['country'] != 'Unspecified'].copy()

# Country statistics
country_stats = df_clean.groupby('country')['customer_aov'].agg(['count', 'mean', 'median', 'std']).reset_index()
country_stats = country_stats.sort_values(by='mean', ascending=False)
country_stats

In [ ]:
# Visualize AOV across Countries using a Boxplot
sns.boxplot(data=df_clean, x='country', y='customer_aov', palette='Set2')
plt.title("Customer AOV Distribution by Country")
plt.xlabel("Country")
plt.ylabel("AOV ($)")
plt.xticks(rotation=45)
plt.show()

### Perform One-Way ANOVA
We fit an ordinary least squares (OLS) model and run ANOVA.

In [ ]:
model = ols('customer_aov ~ C(country)', data=df_clean).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

### Post-Hoc Analysis: Tukey's HSD (Honestly Significant Difference)
If ANOVA is significant (p-value $< 0.05$), we run Tukey's test to determine which specific countries differ.

In [ ]:
tukey = pairwise_tukeyhsd(endog=df_clean['customer_aov'],
                          groups=df_clean['country'],
                          alpha=0.05)
# Convert results to dataframe for clean printing
tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
significant_pairs = tukey_df[tukey_df['reject'] == True]
print(f"Found {significant_pairs.shape[0]} pairs with statistically significant differences in AOV:")
significant_pairs

### Business Interpretation & Actionable Recommendations
1. **High AOV Countries**: We should promote bundled premium products and loyalty memberships in countries with high AOV (e.g., Australia) since customers are willing to make larger transactions.
2. **Low AOV Countries**: In lower AOV countries (e.g., Canada), marketing should focus on discount vouchers, cross-selling cheaper add-ons, or lowering free shipping thresholds to push orders above the current low average.